# Validate the 1st order models (fine-tuned against only fake data) against fake data

This notebook documents the validation of the first-order models (fine-tuned only against fake data) against a test set of fake images. It takes the model checkpoints created by ```finetune_original_models.ipynb```, runs an extraction job for the fake test data from each model (on Azure), downloads the extracted transcriptions, and makes validation summaries comparing the transcriptions with the ground truth for the fake test data.

After completing this notebook, we know how well the first-order models do on the fake data. In particular, we can see improvements compared to the original, zeroth-order models.

In [1]:
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from posixpath import normpath

# Define model settings for validation jobs.
# Each entry: (label, model_slug, batch_size, total_shards)
MODEL_SETTINGS = [
    ("SmolVLM", "smolvlm2", 50, 1),
    ("Granite4", "granite4", 50, 1),
    ("Gemma3", "gemma3", 50, 1),
    ("Gemma4", "gemma4", 50, 1),
    ("Ministral", "ministral", 15, 1),
]

# ND96amsr A100 v4 has 8 GPUs per node. Use one extraction worker per GPU.
NODE_GPU_WORKERS = 8

# Must match the dataset used in finetune_original_models.ipynb
DATASET_NAME = "fake"
DATASET_DIR = "fake_daily_rainfall_2"

if DATASET_DIR:
    TRAINING_IMAGES_PATH = f"{DATASET_DIR.rstrip('/')}/images"
else:
    DATASET_IMAGES_MAP = {
        "real": "Daily_rainfall_sample/images",
        "fake": "fake_daily_rainfall/images",
        "test_real": "test_data/real/images",
        "test_fake": "test_data/fake/images",
    }
    if DATASET_NAME not in DATASET_IMAGES_MAP:
        raise ValueError(f"Unknown DATASET_NAME: {DATASET_NAME}")
    TRAINING_IMAGES_PATH = DATASET_IMAGES_MAP[DATASET_NAME]

print(f"Checkpoint training dataset path: {TRAINING_IMAGES_PATH}")
print(f"Node GPU workers per extraction job: {NODE_GPU_WORKERS}")

Checkpoint training dataset path: fake_daily_rainfall_2/images
Node GPU workers per extraction job: 8


## 1. Auto-discover fine-tuned checkpoints

This cell looks up the latest checkpoint for each model trained on the selected training dataset path.

In [2]:
def _first_existing_path(candidates: list[Path]) -> Path:
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


# Canonical registry location (shared across all notebooks and scripts)
REGISTRY_PATH = Path("../../outputs/model_registry.json")
if not REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Registry not found: {REGISTRY_PATH.resolve()}")


def _norm_rel(path_like: str) -> str:
    return normpath(path_like.replace("\\", "/")).lstrip("./")


def _parse_created_at(value: str) -> datetime:
    if not value:
        return datetime.min.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)


def _fix_checkpoint_path(path: str) -> str:
    """Keep canonical checkpoint path from the registry.
    
    aml_submit.sh resolves this path relative to AML_DATASTORE_BASE.
    Stripping Daily_rainfall_sample/ can break checkpoint mounts when
    the datastore base is rooted above that folder.
    """
    return _norm_rel(path)


registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
# Support both registry schemas (though now we only have one):
# 1) {"checkpoints": [...]} with keys model_slug/training_dataset_path
# 2) {"models": [...]} with keys base_model/dataset
entries = registry.get("checkpoints")
if not isinstance(entries, list):
    entries = registry.get("models", [])

MODEL_JOBS = []
missing = []
expected_dataset_norm = _norm_rel(TRAINING_IMAGES_PATH)

for label, model_slug, batch_size, total_shards in MODEL_SETTINGS:
    candidates = []
    for e in entries:
        entry_model = str(e.get("model_slug", e.get("base_model", "")))
        entry_dataset = str(
            e.get("training_dataset_path", e.get("dataset", ""))
        )
        checkpoint_path = e.get("checkpoint_path")

        if (
            entry_model == model_slug
            and _norm_rel(entry_dataset) == expected_dataset_norm
            and checkpoint_path
        ):
            candidates.append(e)

    if not candidates:
        missing.append(label)
        continue

    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )

    raw_checkpoint_path = str(best["checkpoint_path"])
    checkpoint_path = _fix_checkpoint_path(raw_checkpoint_path)
    MODEL_JOBS.append((label, checkpoint_path, batch_size, total_shards))
    print(f"{label}: {checkpoint_path}")

if missing:
    raise RuntimeError(
        "Missing fine-tuned checkpoints for: "
        + ", ".join(missing)
        + ". Check DATASET_DIR / DATASET_NAME and ensure model_registry has entries for this dataset."
    )

print(f"\nUsing canonical registry: {REGISTRY_PATH.resolve()}")
print(f"Discovered {len(MODEL_JOBS)} checkpoints")

SmolVLM: Daily_rainfall_sample/outputs/checkpoints/smolvlm2-20260625-094936/HuggingFaceTB--SmolVLM2-2.2B-Instruct
Granite4: Daily_rainfall_sample/outputs/checkpoints/granite4-20260625-095008/ibm-granite--granite-vision-4.1-4b
Gemma3: Daily_rainfall_sample/outputs/checkpoints/gemma3-20260625-095027/google--gemma-3-4b-it
Gemma4: Daily_rainfall_sample/outputs/checkpoints/gemma4-20260625-095044/google--gemma-4-E4B-it
Ministral: Daily_rainfall_sample/outputs/checkpoints/ministral-20260625-095102/mistralai--Mistral-Small-3.1-24B-Instruct-2503

Using canonical registry: /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/model_registry.json
Discovered 5 checkpoints


## 2. Run extraction jobs with fine-tuned checkpoints

This submits one extraction job per fine-tuned model checkpoint on fake test images.

In [3]:
for model_name, checkpoint, batch_size, total_shards in MODEL_JOBS:
    print(f"Submitting {model_name}...")
    subprocess.run(
        [
            "bash",
            "../../scripts/aml_submit.sh",
            "--checkpoint",
            checkpoint,
            "--images-path",
            "test_data/fake/images",
            "--transcriptions-path",
            "test_data/fake/transcriptions",
            "--total-shards",
            str(total_shards),
            "--node-gpu-workers",
            str(NODE_GPU_WORKERS),
            "--batch-size",
            str(batch_size),
            "--extraction-registry",
            "../../outputs/extraction_registry.json",
            "extract",
        ],
        check=True,
    )

Submitting SmolVLM...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/fake/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint: azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs/checkpoints/smolvlm2-20260625-094936/HuggingFace

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

goofy_ship_wy231b17hb
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260625-105337
  Model: smolvlm
  Dataset: test_data/fake/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Granite4...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/fake/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint: az

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

patient_cartoon_h5w2z8cdnv
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260625-105408
  Model: smolvlm
  Dataset: test_data/fake/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Gemma3...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/fake/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint:

...

busy_knee_srqy31ksqr
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260625-105427
  Model: smolvlm
  Dataset: test_data/fake/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Gemma4...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/fake/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint: azure

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

eager_forest_tz7zt0b40t
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260625-105443
  Model: smolvlm
  Dataset: test_data/fake/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Ministral...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/fake/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint:

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

bubbly_dress_g9g1xxqr3b
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260625-105500
  Model: smolvlm
  Dataset: test_data/fake/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json


When the jobs have been submitted, the extraction registry will contain run names needed for download and analysis.

Run the next cell to discover run names from the registry, then paste the printed `RUN_NAMES = [...]` block into the following cell.

In [ ]:
# Discover run names from extraction registry after submissions complete.
# This cell does NOT write external files. It prints a block to paste into the next cell.

EXTRACTION_REGISTRY_PATH = Path("../../outputs/extraction_registry.json")
TARGET_IMAGES_PATH = "test_data/fake/images"

registry = json.loads(EXTRACTION_REGISTRY_PATH.read_text(encoding="utf-8"))
entries = registry.get("extractions", [])

RUN_NAMES = []
missing = []
target_images_norm = _norm_rel(TARGET_IMAGES_PATH)

for model_name, checkpoint, _batch_size, _total_shards in MODEL_JOBS:
    candidates = [
        e
        for e in entries
        if _norm_rel(str(e.get("images_path", ""))) == target_images_norm
        and str(e.get("checkpoint_path", "")) == checkpoint
        and e.get("run_name")
    ]
    if not candidates:
        missing.append(model_name)
        continue
    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )
    run_name = str(best["run_name"])
    RUN_NAMES.append(run_name)
    print(f"{model_name}: {run_name}")

if missing:
    raise RuntimeError(
        "Missing extraction runs in registry for: "
        + ", ".join(missing)
        + ". Run extraction submission first, then re-run this cell."
    )

EXTRACTION_DIRS = [f"../outputs/extractions/{r}" for r in RUN_NAMES]

print("\nCopy this block into the next cell:\n")
print("RUN_NAMES = [")
for run_name in RUN_NAMES:
    print(f'    "{run_name}",')
print("]\n")


SmolVLM: 20260625-105337
Granite4: 20260625-105408
Gemma3: 20260625-105427
Gemma4: 20260625-105443
Ministral: 20260625-105500

Copy this block into the next cell:

RUN_NAMES = [
    "20260625-105337",
    "20260625-105408",
    "20260625-105427",
    "20260625-105443",
    "20260625-105500",
]

EXTRACTION_DIRS = [f"../outputs/extractions/{r}" for r in RUN_NAMES]


In [5]:
# Persistent run names for this notebook.
# Paste the RUN_NAMES block printed by the previous cell here.
RUN_NAMES = [
    "20260625-105337",
    "20260625-105408",
    "20260625-105427",
    "20260625-105443",
    "20260625-105500",
]

EXTRACTION_DIRS = [f"../../outputs/extractions/{r}" for r in RUN_NAMES]

if not RUN_NAMES:
    raise RuntimeError("RUN_NAMES is empty. Run the previous cell, then paste its output here.")

print("Using run names:")
for run_name in RUN_NAMES:
    print("  ", run_name)

Using run names:
   20260625-105337
   20260625-105408
   20260625-105427
   20260625-105443
   20260625-105500


When the jobs have completed successfully, download the extractions so we can analyze them locally.

In [7]:
for run_name in RUN_NAMES:
    subprocess.run(
        ["bash", "../../scripts/aml_download.sh", "--run-name", run_name],
        check=True,
    )

Resolving datastore 'large_datastore' in workspace 'mlw-llmdatarescue-uksouth-01'...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default

             to:  /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260625-105337
         workers: 16
           quiet: false
  DRain_1851-1860_Kent-851.json
  DRain_1861-1870_Buckinghamshire-569.json
  DRain_1861-1870_Devon-805.json
  DRain_1861-1870_Northumberland-503.json
  DRain_1861-1870_Worcestershire-331.json
  DRain_1871-1880_Bedfordshire-552.json
  DRain_1871-1880_Bedfordshire-759.json
  DRain_1871-1880_Cambridgeshire-366.json
  DRain_1871-1880_Flintshire-261.json
  DRain_1871-1880_Leicestershire-947.json
  DRain_1871-1880_Middlesex-660.json
  DRain_1871-1880_Suffolk-906.json
  DRain_1881-1890_Hampshire-128.json
  DRain_1881-1890_Northamptonshire-338.json
  DRain_1881-1890_Northumberland-177.json
  DRain_1881-1890_Sussex-6.json
  DRain_1891-1900_Cumberland-171.json
  DRain_1891-1900_Cumberland-405.json
  DRain_1891-1900_Flintshire-492.json
  DRain_1891-1900_Northumberland-518.json
  DRain_18

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default

             to:  /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260625-105408
         workers: 16
           quiet: false
  DRain_1851-1860_Kent-851.json
  DRain_1861-1870_Buckinghamshire-569.json
  DRain_1861-1870_Devon-805.json
  DRain_1861-1870_Northumberland-503.json
  DRain_1861-1870_Worcestershire-331.json
  DRain_1871-1880_Bedfordshire-552.json
  DRain_1871-1880_Bedfordshire-759.json
  DRain_1871-1880_Cambridgeshire-366.json
  DRain_1871-1880_Flintshire-261.json
  DRain_1871-1880_Leicestershire-947.json
  DRain_1871-1880_Middlesex-660.json
  DRain_1871-1880_Suffolk-906.json
  DRain_1881-1890_Hampshire-128.json
  DRain_1881-1890_Northamptonshire-338.json
  DRain_1881-1890_Northumberland-177.json
  DRain_1881-1890_Sussex-6.json
  DRain_1891-1900_Cumberland-171.json
  DRain_1891-1900_Cumberland-405.json
  DRain_1891-1900_Flintshire-492.json
  DRain_1891-1900_Northumberland-518.json
  DRain_18

...

Storage account: sallmdatarescue02  container: default

             to:  /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260625-105427
         workers: 16
           quiet: false
  DRain_1851-1860_Kent-851.json
  DRain_1861-1870_Buckinghamshire-569.json
  DRain_1861-1870_Devon-805.json
  DRain_1861-1870_Northumberland-503.json
  DRain_1861-1870_Worcestershire-331.json
  DRain_1871-1880_Bedfordshire-552.json
  DRain_1871-1880_Bedfordshire-759.json
  DRain_1871-1880_Cambridgeshire-366.json
  DRain_1871-1880_Flintshire-261.json
  DRain_1871-1880_Leicestershire-947.json
  DRain_1871-1880_Middlesex-660.json
  DRain_1871-1880_Suffolk-906.json
  DRain_1881-1890_Hampshire-128.json
  DRain_1881-1890_Northamptonshire-338.json
  DRain_1881-1890_Northumberland-177.json
  DRain_1881-1890_Sussex-6.json
  DRain_1891-1900_Cumberland-171.json
  DRain_1891-1900_Cumberland-405.json
  DRain_1891-1900_Flintshire-492.json
  DRain_1891-1900_Northumberland-518.json
  DRain_18

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default

             to:  /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260625-105443
         workers: 16
           quiet: false
  DRain_1851-1860_Kent-851.json
  DRain_1861-1870_Buckinghamshire-569.json
  DRain_1861-1870_Devon-805.json
  DRain_1861-1870_Northumberland-503.json
  DRain_1861-1870_Worcestershire-331.json
  DRain_1871-1880_Bedfordshire-552.json
  DRain_1871-1880_Bedfordshire-759.json
  DRain_1871-1880_Cambridgeshire-366.json
  DRain_1871-1880_Flintshire-261.json
  DRain_1871-1880_Leicestershire-947.json
  DRain_1871-1880_Middlesex-660.json
  DRain_1871-1880_Suffolk-906.json
  DRain_1881-1890_Hampshire-128.json
  DRain_1881-1890_Northamptonshire-338.json
  DRain_1881-1890_Northumberland-177.json
  DRain_1881-1890_Sussex-6.json
  DRain_1891-1900_Cumberland-171.json
  DRain_1891-1900_Cumberland-405.json
  DRain_1891-1900_Flintshire-492.json
  DRain_1891-1900_Northumberland-518.json
  DRain_18

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default

             to:  /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260625-105500
         workers: 16
           quiet: false
  DRain_1851-1860_Kent-851.json
  DRain_1861-1870_Buckinghamshire-569.json
  DRain_1861-1870_Devon-805.json
  DRain_1861-1870_Northumberland-503.json
  DRain_1861-1870_Worcestershire-331.json
  DRain_1871-1880_Bedfordshire-552.json
  DRain_1871-1880_Bedfordshire-759.json
  DRain_1871-1880_Cambridgeshire-366.json
  DRain_1871-1880_Flintshire-261.json
  DRain_1871-1880_Leicestershire-947.json
  DRain_1871-1880_Middlesex-660.json
  DRain_1871-1880_Suffolk-906.json
  DRain_1881-1890_Hampshire-128.json
  DRain_1881-1890_Northamptonshire-338.json
  DRain_1881-1890_Northumberland-177.json
  DRain_1881-1890_Sussex-6.json
  DRain_1891-1900_Cumberland-171.json
  DRain_1891-1900_Cumberland-405.json
  DRain_1891-1900_Flintshire-492.json
  DRain_1891-1900_Northumberland-518.json
  DRain_18

Build 1st-order consensus from downloaded extractions and compare to ground truth.

This produces outputs in `outputs/consensus_fake_data/consensus_1st_order`.

In [9]:
cmd = [
    "bash",
    "../../scripts/run_consensus_pipeline.sh",
    "--dataset-root",
    "../../outputs/consensus_fake_data",
    "--images-dir",
    "../../test_data/fake/images",
    "--variant-name",
    "test_1st_order",
    "--threshold",
    "3",
    "--null-threshold",
    "5",
    "--precision",
    "3",
]

for extraction_dir in EXTRACTION_DIRS:
    cmd.extend(["--extraction-dir", extraction_dir])

cmd.extend(
    [
        "--validate",
        "--ground-truth-dir",
        "../../test_data/fake/transcriptions",
        "--validation-sample-denominator",
        "1",
    ]
)

subprocess.run(cmd, check=True)

=== Consensus Pipeline ===
Dataset root:      ../../outputs/consensus_fake_data
Variant name:      test_1st_order
Threshold:         3
Null threshold:    5
Precision:         3
Extraction dirs:   5
Validate:          true
Validation sample: 1/1

Creating config...
✓ Created consensus config
  Variant:           test_1st_order
  Dataset root:      /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/consensus_fake_data
  Variant dir:       /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/consensus_fake_data/test_1st_order
  Config file:       /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/consensus_fake_data/test_1st_order/consensus_config.json
  Extraction dirs:   5 model(s)
                     1. /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260625-105337
                     2. /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260625-105408
                     3. /home/u

CompletedProcess(args=['bash', '../../scripts/run_consensus_pipeline.sh', '--dataset-root', '../../outputs/consensus_fake_data', '--images-dir', '../../test_data/fake/images', '--variant-name', 'test_1st_order', '--threshold', '3', '--null-threshold', '5', '--precision', '3', '--extraction-dir', '../../outputs/extractions/20260625-105337', '--extraction-dir', '../../outputs/extractions/20260625-105408', '--extraction-dir', '../../outputs/extractions/20260625-105427', '--extraction-dir', '../../outputs/extractions/20260625-105443', '--extraction-dir', '../../outputs/extractions/20260625-105500', '--validate', '--ground-truth-dir', '../../test_data/fake/transcriptions', '--validation-sample-denominator', '1'], returncode=0)

## 3. Score each individual extraction against ground truth

This computes per-model extraction quality directly against fake ground-truth transcriptions.

In [11]:
labels = [name for name, _checkpoint, _batch, _shards in MODEL_JOBS]

cmd = [
    "python",
    "../../scripts/evaluate_extraction_quality.py",
    "--ground-truth-dir",
    "../../test_data/fake/transcriptions",
    "--output-json",
    "../../outputs/consensus_fake_data/test_1st_order/extraction_quality_summary.json",
]

for label, extraction_dir in zip(labels, EXTRACTION_DIRS):
    cmd.extend(["--extraction-dir", extraction_dir, "--label", label])

subprocess.run(cmd, check=True)

print("\nSaved quality summary to:")
print(Path("../../outputs/consensus_fake_data/test_1st_order/extraction_quality_summary.json").resolve())


EXTRACTION QUALITY SUMMARY
Label         Acc(all) Acc(eval)  Coverage  Correct  Incorrect Cells(eval)   Miss    Bad
----------------------------------------------------------------------------------------
Gemma4          0.9968    0.9968    1.0000    24497         79       24576      0      0
Granite4        0.9958    0.9958    1.0000    24472        104       24576      0      0
SmolVLM         0.9888    0.9888    1.0000    24300        276       24576      0      0
Gemma3          0.9825    0.9825    1.0000    24146        430       24576      0      0
Ministral       0.9801    0.9801    1.0000    24086        490       24576      0      0

Wrote summary JSON: /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/consensus_fake_data/test_1st_order/extraction_quality_summary.json

Saved quality summary to:
/home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/consensus_fake_data/test_1st_order/extraction_quality_summary.json


In [13]:
import json
from pathlib import Path


def _load_summary(path_str: str) -> list[dict]:
    path = Path(path_str)
    if not path.exists():
        raise FileNotFoundError(f"Summary file not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        if "results" in data and isinstance(data["results"], list):
            return data["results"]
        if "rows" in data and isinstance(data["rows"], list):
            return data["rows"]
    if isinstance(data, list):
        return data
    raise ValueError(f"Unexpected summary format in {path}")


def _index_by_label(rows: list[dict]) -> dict[str, dict]:
    out = {}
    for r in rows:
        label = str(r.get("label", "")).strip()
        if label:
            out[label] = r
    return out


summary_0th = _load_summary("../../outputs/consensus_fake_data/test_0th_order/extraction_quality_summary.json")
summary_1st = _load_summary("../../outputs/consensus_fake_data/test_1st_order/extraction_quality_summary.json")

by_0 = _index_by_label(summary_0th)
by_1 = _index_by_label(summary_1st)
labels = sorted(set(by_0.keys()) | set(by_1.keys()))

metrics = [
    "accuracy_vs_all_ground_truth_cells",
    "accuracy_on_evaluated_cells",
    "coverage_of_ground_truth_cells",
]

print("0TH vs 1ST ORDER EXTRACTION QUALITY")
print("=" * 120)
header = (
    f"{'Model':<12} "
    f"{'Acc(all) 0th':>12} {'Acc(all) 1st':>12} {'Delta':>10} "
    f"{'Acc(eval) 0th':>14} {'Acc(eval) 1st':>14} {'Delta':>10} "
    f"{'Coverage 0th':>12} {'Coverage 1st':>12} {'Delta':>10}"
)
print(header)
print("-" * len(header))

for label in labels:
    r0 = by_0.get(label, {})
    r1 = by_1.get(label, {})

    def _val(row: dict, key: str):
        v = row.get(key)
        return float(v) if isinstance(v, (int, float)) else None

    a0 = _val(r0, metrics[0])
    a1 = _val(r1, metrics[0])
    e0 = _val(r0, metrics[1])
    e1 = _val(r1, metrics[1])
    c0 = _val(r0, metrics[2])
    c1 = _val(r1, metrics[2])

    def _fmt(v):
        return f"{v:0.4f}" if isinstance(v, float) else "N/A"

    def _d(v1, v0):
        if isinstance(v1, float) and isinstance(v0, float):
            return v1 - v0
        return None

    print(
        f"{label:<12} "
        f"{_fmt(a0):>12} {_fmt(a1):>12} {_fmt(_d(a1, a0)):>10} "
        f"{_fmt(e0):>14} {_fmt(e1):>14} {_fmt(_d(e1, e0)):>10} "
        f"{_fmt(c0):>12} {_fmt(c1):>12} {_fmt(_d(c1, c0)):>10}"
    )

print("\nLegend: Delta = 1st_order - 0th_order")
print("Positive deltas indicate improvement in 1st-order over 0th-order.")

0TH vs 1ST ORDER EXTRACTION QUALITY
Model        Acc(all) 0th Acc(all) 1st      Delta  Acc(eval) 0th  Acc(eval) 1st      Delta Coverage 0th Coverage 1st      Delta
-------------------------------------------------------------------------------------------------------------------------------
Gemma3             0.2269       0.9825     0.7556         0.2421         0.9825     0.7404       0.9375       1.0000     0.0625
Gemma4             0.2787       0.9968     0.7181         0.3075         0.9968     0.6893       0.9062       1.0000     0.0938
Granite4           0.8677       0.9958     0.1281         0.8677         0.9958     0.1281       1.0000       1.0000     0.0000
Ministral          0.6038       0.9801     0.3762         0.6038         0.9801     0.3762       1.0000       1.0000     0.0000
SmolVLM            0.2061       0.9888     0.7827         0.2162         0.9888     0.7725       0.9531       1.0000     0.0469

Legend: Delta = 1st_order - 0th_order
Positive deltas indicate impr